In [ ]:
import time
from os import listdir
from os.path import isfile, join

import pandas as pd

from cvrptw.io import calculate_distances, read_instance_data, save_solution
from cvrptw.model import get_distance
from cvrptw.solver import get_greedy_solution, ils, ls_attempts_and_time_limit
from cvrptw.viz import draw_best_solutions

## Benchmark

In [ ]:
INSTANCES_DIR = './resources/instances/'
instances = [
    f for f in listdir(INSTANCES_DIR)
    if isfile(join(INSTANCES_DIR, f)) and (f.endswith('.txt') or f.endswith('.TXT'))
]

perturbation_max_moves = 5
results = []

In [ ]:
for instance_file in instances:
    print()
    print('*' * 75)
    print(instance_file)

    n_vehicles, capacity, customers = read_instance_data(INSTANCES_DIR + instance_file)
    distances = calculate_distances(customers)
    ls_max_moves, ils_time_limit = ls_attempts_and_time_limit(n_vehicles, len(customers))

    start = time.time()
    init_sol = get_greedy_solution(customers, distances, n_vehicles, capacity)
    n_iters, sol = ils(init_sol, ls_max_moves, perturbation_max_moves, ils_time_limit)
    elapsed = time.time() - start

    best_dist = round(get_distance(sol), 2)
    print(f'Best distance = {best_dist}')
    print(f'{elapsed:.2f}/{ils_time_limit:.2f} sec.')

    save_solution('./results/' + instance_file[:-3] + 'sol', sol)
    results.append([instance_file, best_dist, len(sol), n_iters, round(elapsed, 2), sol])

## Results

In [ ]:
columns = ['name', 'distance', 'n_vehicles', 'done_iters', 'time', 'solution']
res_df = pd.DataFrame(results, columns=columns)
res_df

In [ ]:
draw_best_solutions(res_df[['name', 'distance', 'n_vehicles', 'solution']].values)